In [2]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# Setup headless Chrome
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)

# Load your Excel file
df = pd.read_excel("aiib_all_38_pages.xlsx")

# New fields to scrape
df["Project Title"] = ""
df["Project Number"] = ""
df["E&S Category"] = ""
df["Approved Funding"] = ""
df["Financing Type"] = ""
df["Concept Review"] = ""
df["Appraisal Review"] = ""
df["Financing Approval"] = ""
df["Latest Field Visit"] = ""
df["Objective"] = ""
df["Description"] = ""
df["Team Leader"] = ""
df["Implementing Entity"] = ""

# Helper functions
def get_summary_value(label):
    try:
        return driver.find_element(By.XPATH, f"//div[@class='summary-item-name' and normalize-space(text())='{label}']/following-sibling::div").text.strip()
    except:
        return ""

def get_paragraph_text(header):
    try:
        return driver.find_element(By.XPATH, f"//h2[normalize-space(text())='{header}']/following-sibling::p").text.strip()
    except:
        return ""

def get_block_text(header):
    try:
        container = driver.find_element(By.XPATH, f"//h2[normalize-space(text())='{header}']/following-sibling::div")
        return container.text.strip()
    except:
        return ""

# Scrape each row
for index, row in df.iterrows():
    url = row["Detail Link"]
    print(f"🔍 Scraping: {url}")
    try:
        driver.get(url)
        time.sleep(2)

        # Extract all fields
        df.at[index, "Project Title"] = driver.find_element(By.CLASS_NAME, "project-name").text.strip()
        df.at[index, "Project Number"] = get_summary_value("PROJECT NUMBER")
        df.at[index, "E&S Category"] = get_summary_value("E&S CATEGORY")
        df.at[index, "Approved Funding"] = get_summary_value("APPROVED FUNDING")
        df.at[index, "Financing Type"] = get_summary_value("FINANCING TYPE")
        df.at[index, "Concept Review"] = get_summary_value("CONCEPT REVIEW")
        df.at[index, "Appraisal Review"] = get_summary_value("APPRAISAL REVIEW/FINAL REVIEW")
        df.at[index, "Financing Approval"] = get_summary_value("FINANCING APPROVAL")
        df.at[index, "Latest Field Visit"] = get_summary_value("LATEST FIELD VISIT")
        df.at[index, "Objective"] = get_paragraph_text("OBJECTIVE")
        df.at[index, "Description"] = get_paragraph_text("DESCRIPTION")
        df.at[index, "Team Leader"] = get_block_text("PROJECT TEAM LEADER")
        df.at[index, "Implementing Entity"] = get_block_text("IMPLEMENTING ENTITY")

    except Exception as e:
        print(f"⚠️ Failed to scrape {url}: {e}")
        continue

driver.quit()

# Save to new file
df.to_excel("aiib_projects_with_full_details1.xlsx", index=False)
print("✅ Done! All details saved to 'aiib_projects_with_full_details1.xlsx'")


🔍 Scraping: https://www.aiib.org/en/projects/details/2025/proposed/kazakhstan-inclusive-and-sustainable-economic-growth-development-policy-operation.html
🔍 Scraping: https://www.aiib.org/en/projects/details/2025/proposed/uzbekistan-telecom.html
🔍 Scraping: https://www.aiib.org/en/projects/details/2023/approved/India-Manipur-Urban-Road-Drainage-and-Asset-Management-Improvement-Project.html
🔍 Scraping: https://www.aiib.org/en/projects/details/2024/approved/Georgia-Tbilisi-Metro-Modernization-Project.html
🔍 Scraping: https://www.aiib.org/en/projects/details/2025/approved/brazil-sicredi-green-loan.html
🔍 Scraping: https://www.aiib.org/en/projects/details/2024/approved/multicountry-green-energy-capacity-expansion.html
🔍 Scraping: https://www.aiib.org/en/projects/details/2025/approved/bangladesh-north-dhaka-waste-to-energy-project.html
🔍 Scraping: https://www.aiib.org/en/projects/details/2025/approved/bangladesh-climate-resilient-inclusive-development-program-subprogram-2.html
🔍 Scraping: ht

In [9]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

driver.get("https://www.aiib.org/en/projects/list/index.html")

projects = []
MAX_PAGES = 38

for page in range(1, MAX_PAGES + 1):
    print(f"🔄 Scraping page {page}...")
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ul.table-row.row")))
    rows = driver.find_elements(By.CSS_SELECTOR, "ul.table-row.row")

    for row in rows:
        try:
            project = {
                "APPROVAL YEAR": row.find_element(By.CLASS_NAME, "data1").text.strip(),
                "MEMBER": row.find_element(By.CLASS_NAME, "data2").text.strip(),
                "SECTOR": row.find_element(By.CLASS_NAME, "data3").text.strip(),
                "FINANCING TYPE": row.find_element(By.CLASS_NAME, "data5").text.strip(),
                "PROJECT NAME": row.find_element(By.CLASS_NAME, "data6").text.strip(),
                "FINANCING AMOUNT": row.find_element(By.CLASS_NAME, "data7").text.strip(),
                "STATUS": row.find_element(By.CLASS_NAME, "data8").text.strip(),
                "DETAIL LINK": ""
            }

            try:
                link = row.find_element(By.CSS_SELECTOR, "a.viewLink").get_attribute("href")
                if link.startswith("/"):
                    link = "https://www.aiib.org" + link
                project["DETAIL LINK"] = link
            except:
                pass

            projects.append(project)
        except Exception as e:
            print(f"⚠️ Skipping row: {e}")
            continue

    if page < MAX_PAGES:
        try:
            # Locate the pagination links (only visible pages)
            page_links = driver.find_elements(By.CSS_SELECTOR, "div.holder a")

            # Find the correct one by visible text
            for link in page_links:
                if link.text.strip() == str(page + 1):
                    driver.execute_script("arguments[0].click();", link)
                    break

            time.sleep(3)  # Let the new page load
        except Exception as e:
            print(f"❌ Failed to go to page {page + 1}: {e}")
            break

driver.quit()

# Save to Excel
df = pd.DataFrame(projects)
df.to_excel("aiib_all_projects_38pages.xlsx", index=False)
print(f"✅ DONE: {len(projects)} projects saved.")


🔄 Scraping page 1...
🔄 Scraping page 2...
🔄 Scraping page 3...
🔄 Scraping page 4...
🔄 Scraping page 5...
🔄 Scraping page 6...
🔄 Scraping page 7...
🔄 Scraping page 8...
🔄 Scraping page 9...
🔄 Scraping page 10...
🔄 Scraping page 11...
🔄 Scraping page 12...
🔄 Scraping page 13...
🔄 Scraping page 14...
🔄 Scraping page 15...
🔄 Scraping page 16...
🔄 Scraping page 17...
🔄 Scraping page 18...
🔄 Scraping page 19...
🔄 Scraping page 20...
🔄 Scraping page 21...
🔄 Scraping page 22...
🔄 Scraping page 23...
🔄 Scraping page 24...
🔄 Scraping page 25...
🔄 Scraping page 26...
🔄 Scraping page 27...
🔄 Scraping page 28...
🔄 Scraping page 29...
🔄 Scraping page 30...
🔄 Scraping page 31...
🔄 Scraping page 32...
🔄 Scraping page 33...
🔄 Scraping page 34...
🔄 Scraping page 35...
🔄 Scraping page 36...
🔄 Scraping page 37...
🔄 Scraping page 38...
✅ DONE: 14402 projects saved.


In [10]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup Chrome in headless mode
options = Options()
options.add_argument("--headless")  # You can remove this line to see the browser
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)

# Open the AIIB projects page
driver.get("https://www.aiib.org/en/projects/list/index.html")
wait = WebDriverWait(driver, 15)

projects = []

# Loop through all 38 pages
for page_num in range(1, 39):
    print(f"🔄 Scraping page {page_num}...")

    # Wait until rows are loaded
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ul.table-row.row")))
    rows = driver.find_elements(By.CSS_SELECTOR, "ul.table-row.row")

    for row in rows:
        try:
            project_name = row.find_element(By.CLASS_NAME, "data6").text.strip()
            detail_link = ""
            try:
                detail_link = row.find_element(By.CSS_SELECTOR, "a.viewLink").get_attribute("href")
            except:
                pass

            projects.append({
                "Approval Year": row.find_element(By.CLASS_NAME, "data1").text.strip(),
                "Economy": row.find_element(By.CLASS_NAME, "data2").text.strip(),
                "Sector": row.find_element(By.CLASS_NAME, "data3").text.strip(),
                "Financing Type": row.find_element(By.CLASS_NAME, "data5").text.strip(),
                "Project Name": project_name,
                "Financing Amount": row.find_element(By.CLASS_NAME, "data7").text.strip(),
                "Status": row.find_element(By.CLASS_NAME, "data8").text.strip(),
                "Detail Link": f"https://www.aiib.org{detail_link}" if detail_link.startswith("/") else detail_link
            })
        except Exception as e:
            print("⚠️ Skipping row due to error:", e)
            continue

    # Go to the next page
    if page_num < 38:
        try:
            page_links = driver.find_elements(By.CSS_SELECTOR, "div.holder a")
            for link in page_links:
                if link.text.strip() == str(page_num + 1):
                    driver.execute_script("arguments[0].click();", link)
                    time.sleep(3)  # Allow next page to load
                    break
        except Exception as e:
            print(f"❌ Failed to click page {page_num + 1}: {e}")
            break

driver.quit()

# Save data to Excel
df = pd.DataFrame(projects)
df.to_excel("aiib_all_38_pages.xlsx", index=False)
print(f"✅ Done! {len(projects)} projects saved to 'aiib_all_38_pages.xlsx'")


🔄 Scraping page 1...
🔄 Scraping page 2...
🔄 Scraping page 3...
🔄 Scraping page 4...
🔄 Scraping page 5...
🔄 Scraping page 6...
🔄 Scraping page 7...
🔄 Scraping page 8...
🔄 Scraping page 9...
🔄 Scraping page 10...
🔄 Scraping page 11...
🔄 Scraping page 12...
🔄 Scraping page 13...
🔄 Scraping page 14...
🔄 Scraping page 15...
🔄 Scraping page 16...
🔄 Scraping page 17...
🔄 Scraping page 18...
🔄 Scraping page 19...
🔄 Scraping page 20...
🔄 Scraping page 21...
🔄 Scraping page 22...
🔄 Scraping page 23...
🔄 Scraping page 24...
🔄 Scraping page 25...
🔄 Scraping page 26...
🔄 Scraping page 27...
🔄 Scraping page 28...
🔄 Scraping page 29...
🔄 Scraping page 30...
🔄 Scraping page 31...
🔄 Scraping page 32...
🔄 Scraping page 33...
🔄 Scraping page 34...
🔄 Scraping page 35...
🔄 Scraping page 36...
🔄 Scraping page 37...
🔄 Scraping page 38...
✅ Done! 14402 projects saved to 'aiib_all_38_pages.xlsx'


In [11]:
def main():
    # ... your scraping code that populates df ...

    df.to_csv(output_csv_path, index=False)
    print(f"Saved updated data with PDF snippets to {output_csv_path}")


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# Setup Chrome options (optional: uncomment headless to run without UI)
options = Options()
# options.add_argument("--headless")  # Run in headless mode if you want

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

url = "https://www.aiib.org/en/projects/list/index.html"
driver.get(url)

all_projects = []

def parse_projects_on_page():
    projects = []
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ul.table-row.row")))
    rows = driver.find_elements(By.CSS_SELECTOR, "ul.table-row.row")

    for row in rows:
        try:
            approval_year = row.find_element(By.CSS_SELECTOR, "li:nth-child(1) .data1").text.strip()
            economy = row.find_element(By.CSS_SELECTOR, "li:nth-child(2) .data2").text.strip()
            sector = row.find_element(By.CSS_SELECTOR, "li:nth-child(3) .data3").text.strip()
            financing_type = row.find_element(By.CSS_SELECTOR, "li:nth-child(4) .data5").text.strip()

            name_elem = row.find_element(By.CSS_SELECTOR, "li:nth-child(5) .data6")
            project_name = name_elem.text.strip()

            # Safely get the detail URL from any 'VIEW DETAILS' link in the row
            try:
                detail_link = row.find_element(By.CSS_SELECTOR, "a.viewLink")
                detail_url = detail_link.get_attribute("href")
            except:
                detail_url = ""

            financing_amount = row.find_element(By.CSS_SELECTOR, "li:nth-child(6) .data7").text.strip()
            status = row.find_element(By.CSS_SELECTOR, "li:nth-child(7) .data8").text.strip()

            projects.append({
                "Approval Year": approval_year,
                "Economy": economy,
                "Sector": sector,
                "Financing Type": financing_type,
                "Project Name": project_name,
                "Project Details URL": detail_url,
                "Financing Amount": financing_amount,
                "Status": status
            })
        except Exception as e:
            print(f"Error parsing a row: {e}")
            continue
    return projects

while True:
    print(f"Scraping page...")
    page_projects = parse_projects_on_page()
    all_projects.extend(page_projects)

    # Try to click the "next" button, stop if not available or disabled
    try:
        next_button = driver.find_element(By.CSS_SELECTOR, "a.jpg-next")
        if "disabled" in next_button.get_attribute("class"):
            print("Last page reached.")
            break
        else:
            next_button.click()
            time.sleep(2)  # wait for page to load
    except Exception as e:
        print("No next button or last page reached.")
        break

driver.quit()

# Save all data to Excel
df = pd.DataFrame(all_projects)
df.to_excel("aiib_projects_all.xlsx", index=False)
print(f"Scraped {len(all_projects)} projects saved to aiib_projects_all.xlsx")


Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Scraping page...
Last page reached.
Scraped 14440 projects saved to aiib_projects_all.xlsx


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

options = Options()
# options.add_argument("--headless")  # Uncomment for headless mode

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

url = "https://www.aiib.org/en/projects/list/index.html"
driver.get(url)

all_projects = []

def parse_projects_on_page():
    projects = []
    # Wait for the rows to be visible
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ul.table-row.row")))
    rows = driver.find_elements(By.CSS_SELECTOR, "ul.table-row.row")

    for row in rows:
        try:
            approval_year = row.find_element(By.CSS_SELECTOR, "li:nth-child(1) .data1").text.strip()
            economy = row.find_element(By.CSS_SELECTOR, "li:nth-child(2) .data2").text.strip()
            sector = row.find_element(By.CSS_SELECTOR, "li:nth-child(3) .data3").text.strip()
            financing_type = row.find_element(By.CSS_SELECTOR, "li:nth-child(4) .data5").text.strip()
            project_name = row.find_element(By.CSS_SELECTOR, "li:nth-child(5) .data6").text.strip()

            try:
                detail_link = row.find_element(By.CSS_SELECTOR, "a.viewLink")
                detail_url = detail_link.get_attribute("href")
            except:
                detail_url = ""

            financing_amount = row.find_element(By.CSS_SELECTOR, "li:nth-child(6) .data7").text.strip()
            status = row.find_element(By.CSS_SELECTOR, "li:nth-child(7) .data8").text.strip()

            projects.append({
                "Approval Year": approval_year,
                "Economy": economy,
                "Sector": sector,
                "Financing Type": financing_type,
                "Project Name": project_name,
                "Project Details URL": detail_url,
                "Financing Amount": financing_amount,
                "Status": status
            })
        except Exception as e:
            print(f"Error parsing a row: {e}")
    return projects


def get_current_page_number():
    # The current page number has class jp-current
    try:
        current_page = driver.find_element(By.CSS_SELECTOR, "a.jp-current").text
        return int(current_page)
    except:
        return 1


def get_total_pages():
    # Find all page number links except the ... hidden ones and get the max number
    page_links = driver.find_elements(By.CSS_SELECTOR, "div.holder a")
    page_numbers = []
    for link in page_links:
        text = link.text.strip()
        if text.isdigit():
            page_numbers.append(int(text))
    if page_numbers:
        return max(page_numbers)
    else:
        return 1


def click_page(page_num):
    # Click on page link with given page_num
    page_links = driver.find_elements(By.CSS_SELECTOR, "div.holder a")
    for link in page_links:
        if link.text.strip() == str(page_num):
            link.click()
            return True
    return False


# First get total pages
total_pages = get_total_pages()
print(f"Total pages: {total_pages}")

for page in range(1, total_pages + 1):
    print(f"Scraping page {page}...")
    # If not first page, click to go to that page
    if page > 1:
        clicked = click_page(page)
        if not clicked:
            print(f"Could not click page {page}, exiting.")
            break
        # Wait until new page loads by waiting for first row's Approval Year to update or presence of rows
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ul.table-row.row")))
        # A small sleep to ensure complete loading
        time.sleep(2)

    page_projects = parse_projects_on_page()
    all_projects.extend(page_projects)

driver.quit()

df = pd.DataFrame(all_projects)
df.to_excel("aiib_projects_all_fixed.xlsx", index=False)
print(f"Scraped {len(all_projects)} projects saved to aiib_projects_all_fixed.xlsx")


Total pages: 38
Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraped 14440 projects saved to aiib_projects_all_fixed.xlsx
